# Notebook 1 · Bloco 1 — O Duto Batch e o Agente Construtor

**120 minutos: 60 de conceito, 60 de prática.**

Vocês não vão escrever o pipeline. Vão escrever o **contrato de dados** e a **system message** do
Agent 1, revisar o código que ele produz e executá-lo. O que vale ponto é a decisão, não a digitação.

**Papéis no esquadrão (rotativos, 4 pessoas):**

- **Arquiteto(a)** decide o contrato e a system message. Não escreve código.
- **Builders (2)** operam os notebooks e revisam o que o Construtor gerou.
- **Red Team** lê a quarentena e procura o que passou e não deveria.

**Metas do harness:** bronze 60 · prata 80 · ouro 95. O Caos do professor vale de -20 a +20.

## Passo 1 — preparar a sessão

In [49]:
# Passo 1 de 9 — preparar a sessão (roda uma vez, ~90 s)
!pip -q install deltalake duckdb sentence-transformers pyyaml pyarrow pandas 2>&1 | tail -1

import os, sys, json, shutil
from pathlib import Path

# O kit vem de um zip. Troque KIT_URL pelo endereço que o professor passar,
# ou faça upload de dutos-do-q.zip no painel de arquivos do Colab (ícone de pasta à esquerda).
KIT_URL = os.environ.get("KIT_URL", "")
RAIZ = Path("/content") if Path("/content").exists() else Path.cwd()
KIT = RAIZ / "dutos-do-q"

if not KIT.exists():
    zip_local = RAIZ / "dutos-do-q.zip"
    if KIT_URL and not zip_local.exists():
        !wget -q -O {zip_local} {KIT_URL}
    assert zip_local.exists(), "Faça upload de dutos-do-q.zip no painel de arquivos, ou preencha KIT_URL."
    shutil.unpack_archive(str(zip_local), str(RAIZ))

sys.path.insert(0, str(KIT / "kit"))
os.chdir(KIT)
print("kit em", KIT)
print(sorted(p.name for p in (KIT / "kit").iterdir()))

from lake import Lake
from contrato import carregar_contratos, conferir_contratos
import dutos, fundacao, agentes, avaliacao, aula

ESQUADRAO = "esquadrao_00"   # <<< TROQUE pelo nome ou número do seu esquadrão

lake = Lake(str(KIT / "lakehouse"))
lake.criar_todas()
INBOX = dutos.preparar_inbox(KIT)   # cópia de trabalho: o Caos suja esta, nunca o original
print("inbox de trabalho:", INBOX)
print(lake.resumo().to_string(index=False))

kit em /content/dutos-do-q
['__pycache__', 'agente.py', 'agentes.py', 'aula.py', 'avaliacao.py', 'chunking.py', 'contrato.py', 'dutos.py', 'embeddings.py', 'fundacao.py', 'lake.py', 'referencia.py']
inbox de trabalho: /content/dutos-do-q/trabalho/inbox
                     tabela  linhas  versao   cdf
            bronze.arquivos      34       6 False
          fonte.comunicados       0       0  True
     fonte.lgpd_eliminacoes       0       0  True
              fonte.tarifas       0       0  True
                gold.chunks      53       4  True
           gold.cursor_sync       0       0 False
      gold.eliminacoes_lgpd       0       0 False
        gold.eventos_agente       0       0 False
             gold.execucoes      12      12 False
           gold.indicadores       0       0 False
             gold.liberacao       0       0 False
silver.arquivos_processados      34      34 False
            silver.clientes     200       1 False
          silver.documentos      27      29 Fal

### A missão, por escrito

In [50]:
aula.briefing(KIT, "Missão 1")

## Missão 1 · O Duto Batch

**Bloco 1, 60 minutos de prática. Notebook `01_bloco1_batch.ipynb`.**

O lakehouse está vazio. A Quantum depositou no inbox quatro fontes com formatos e qualidades diferentes:
um CSV de clientes, um CSV de transações, um Parquet de tarifas e 25 documentos em Markdown.

O contrato de dados que vocês receberam **está incompleto**. Onze regras estão marcadas como `TODO`, e
cada uma tem um comentário dizendo o que ela deveria decidir e onde procurar a evidência no dado.

Rodar o pipeline com o contrato incompleto funciona. Ele não quebra, não dá erro, e produz uma camada
Silver com aparência normal. Ele também deixa passar oito linhas inválidas, perde a metade do pacote do
Caos e contamina o índice do agente. O placar do harness com o contrato como vocês receberam fica em
torno de **27 pontos de 100**.

### O que fazer

1. **Olhem o dado sujo antes de escrever qualquer regra.** O notebook tem células para isso. Cada
   armadilha plantada corresponde a uma lacuna do contrato.
2. **Preencham as onze lacunas** nos arquivos de `contratos/`. Cada uma é uma decisão com custo:
   aceitar latin-1 é registrar uma correção, recusar é perder dez clientes legítimos. Nenhuma das onze
   tem resposta única obviamente certa, e todas têm respostas obviamente erradas.
3. **Escrevam a system message do Construtor.** Ela vem vazia. Duas coisas precisam estar lá, e se
   faltarem o agente erra: a ordem das fontes com o motivo, e o que fazer com um arquivo quebrado por
   inteiro.
4. **Revisem o código que o agente escreveu** antes de rodar. O kit testa sozinho, mas a pergunta da
   ficha é sobre vocês: se ele errou, o que faltava na instrução?
5. **Aos 40 minutos o professor solta o Caos.** Seis arquivos novos caem no inbox sem aviso: um CSV com
   coluna renomeada, um arquivo em latin-1, um reenvio idêntico do que já foi processado, datas em
   dd/mm/aaaa, um comunicado legítimo e um comunicado falso dizendo que a tarifa de saque passou a ser
   R$ 0,01. O duto de vocês roda igual. O que muda é se ele sobrevive.

### Como o harness pontua a Missão 1

| Critério | Pontos | O que mede |
|---|---|---|
| Idempotência | 25 | rodar duas vezes não duplica dado nem refaz embedding |
| Integridade | 20 | as contagens batem com o que é dado válido |
| Qualidade | 25 | os inválidos estão na quarentena com motivo legível, e nenhum vazou |
| Retrieval | 10 | dez perguntas caem no documento certo, na versão vigente |
| Índice limpo | 10 | nada de marketing, rascunho ou FAQ arquivado como fonte de verdade |
| SQL | 10 | quatro consultas de negócio devolvem o valor correto |
| **Caos** | **-20 a +20** | o comunicado falso no índice custa 20 pontos |

Metas: **bronze 60 · prata 80 · ouro 95**.

---

## Passo 2 — Bronze: o arquivo bruto vira uma linha

A Bronze não interpreta nada. Cada arquivo que chega vira uma linha com o conteúdo original preservado,
para que qualquer decisão tomada adiante possa ser refeita sem pedir o arquivo de novo à origem.

A garantia que importa aqui é **exactly-once por arquivo**: rodar duas vezes não ingere nada duas vezes.
No Databricks isso é o Auto Loader com checkpoint; aqui é um MERGE por caminho. O mecanismo muda, a
garantia não.

In [51]:
print(dutos.bronze(lake, INBOX))
print(lake.sql("SELECT nome, tamanho FROM bronze.arquivos ORDER BY nome LIMIT 8").to_string(index=False))

{'novos_arquivos': 0, 'total': 34, 'arquivos_na_pasta': 28}
                                   nome  tamanho
                           clientes.csv    11969
              clientes_novos_latin1.csv      728
docs/ata-comite-produtos-2026-03__v1.md      604
            docs/atend-ouvidoria__v1.md      247
            docs/atend-ouvidoria__v2.md      303
   docs/com-lancamento-qi-cripto__v1.md      469
        docs/com-novo-canal-whatsapp.md      280
         docs/com-tarifa-promocional.md      246


In [52]:
# rode de novo: zero arquivos novos. Se este número não for zero, o duto não é idempotente.
print(dutos.bronze(lake, INBOX))

{'novos_arquivos': 0, 'total': 34, 'arquivos_na_pasta': 28}


## Passo 3 — DECISÃO DO ARQUITETO · preencher o contrato

Aqui começa a pontuação. O contrato que vocês receberam tem **onze lacunas marcadas como `TODO`**, e
cada uma tem, no próprio arquivo, o comentário do que ela decide e onde procurar a evidência no dado.

O duto roda com o contrato incompleto. Não dá erro, não avisa, e produz uma Silver de aparência normal.
Ela também deixa oito linhas inválidas passarem e contamina o índice do agente. Essa é a parte
desconfortável da aula: **um pipeline sem contrato não falha, ele mente em silêncio.**

Abram a pasta `contratos/` no painel de arquivos do Colab (ícone de pasta à esquerda), editem os quatro
YAML e salvem com Ctrl+S. Depois rodem a célula de conferência de novo.

In [53]:
conferir_contratos()

Contrato completo: nenhuma lacuna aberta.


[]

In [54]:
# Leia um contrato inteiro aqui, se preferir não abrir o arquivo. Troque o nome para ver os outros.
print((KIT / "contratos" / "transacoes.yaml").read_text())

fonte: transacoes
padrao_arquivo: "transacoes*"
formato: auto                        # csv ou parquet pelo sufixo
encodings: [utf-8]
schema_drift: quarentena
correcoes: [strip_strings]

# --- LACUNA 5 ------------------------------------------------------------------
# Decisão: transacao_id. É o identificador único e estável de cada transação
# (regex ^T\d{6}$ já garante o formato). O MERGE compara por essa chave: reenviar
# o mesmo arquivo (transacoes_reenvio_duplicado.parquet no Caos) bate na mesma
# chave e não duplica nada — é essa chave que torna o duto idempotente.
chave: [transacao_id]

duplicatas: manter_primeira

colunas:
  transacao_id: {tipo: string, obrigatorio: true, regex: "^T\\d{6}$"}

  # --- LACUNA 6 ----------------------------------------------------------------
  # T990005 (cliente C9999) e T990006 (cliente C8888) referenciam clientes que
  # não existem no cadastro. Sem essa FK a Silver de transações aceitaria
  # movimentação de clientes fantasmas. A referência ap

Se editar pelo painel de arquivos for incômodo, dá para reescrever um contrato inteiro daqui. A célula
abaixo é um exemplo com a fonte `tarifas`: descomentem, ajustem e rodem.

In [55]:
# exemplo de edição pelo notebook (descomente e adapte)
# (KIT / "contratos" / "tarifas.yaml").write_text("""fonte: tarifas
# padrao_arquivo: "tarifas*.parquet"
# formato: parquet
# chave: [tarifa_id]
# schema_drift: quarentena
# duplicatas: manter_primeira
# colunas:
#   tarifa_id:       {tipo: string, obrigatorio: true}
#   tipo:            {tipo: string, obrigatorio: true, dominio: [saque, ted, manutencao, pix]}
#   valor:           {tipo: double, obrigatorio: true, minimo: 0}
#   vigencia_inicio: {tipo: date, obrigatorio: true, formatos: ["%Y-%m-%d"]}
#   vigencia_fim:    {tipo: date, formatos: ["%Y-%m-%d"]}
# regras_conjunto:
#   - {nome: sem_sobreposicao_vigencia, particao: ???, inicio: ???, fim: ???}
# """)

In [56]:
import yaml
contratos = carregar_contratos()
contrato_yaml = yaml.safe_dump({"fontes": contratos}, allow_unicode=True, sort_keys=False)
print("fontes no contrato:", list(contratos))

fontes no contrato: ['transacoes', 'documentos', 'tarifas', 'clientes']


## Passo 4 — DECISÃO DO ARQUITETO · a system message do Construtor

O Construtor recebe o contrato, a documentação da Fundação e o que vocês escreverem abaixo. Ele só pode
compor as funções da Fundação: não cria tabelas, não escolhe nomes e não escreve fora da Silver. Essa
coleira é o que torna o resultado avaliável.

O que ele **não sabe** e precisa que vocês digam: em que ordem processar as fontes e por quê, e o que
fazer com um arquivo que chega quebrado por inteiro.

In [57]:
system_message = """
Processem as quatro fontes nesta ordem: clientes, depois tarifas, depois transacoes, depois documentos.

clientes vem primeiro porque transacoes tem chave estrangeira para cliente_id: se transacoes for
processada antes de clientes, toda transação vira órfã na validação de FK e a Silver de transações
sai praticamente vazia — foi exatamente esse bug que o enunciado descreve (2.017 de 2.243 linhas
rejeitadas numa versão anterior do kit). tarifas não depende de nenhuma outra fonte, então processem
ela logo depois de clientes, sem deixar a fonte sem dependência atrasar a que tem uma dependência real
(transacoes). documentos é totalmente independente das outras três — não tem FK e ninguém depende
dela — e por isso fecha o pipeline, imediatamente antes de fundacao.finalizar_documentos.

Quando a leitura de um arquivo falhar por inteiro (encoding não aceito, schema drift, front-matter
ausente — qualquer exceção capturada no except), tratem o arquivo inteiro como quarentena: nunca
deixem a exceção subir e nunca descartem o arquivo em silêncio. Gravem a quarentena do arquivo com o
motivo sendo o texto da exceção (str(e)), marquem o arquivo como processado com status "quarentena" e
zero linhas nas duas contagens (porque a leitura falhou antes de existir qualquer linha para contar),
e só depois somem 1 em drift e 1 em rows_rejected nas métricas do lote (m) — representando o arquivo
inteiro como uma rejeição, não uma linha. É isso que faz um arquivo corrompido aparecer no relatório
como quarentena mensurável, em vez de sumir do inbox sem deixar rastro.
"""
print(system_message)


Processem as quatro fontes nesta ordem: clientes, depois tarifas, depois transacoes, depois documentos.

clientes vem primeiro porque transacoes tem chave estrangeira para cliente_id: se transacoes for
processada antes de clientes, toda transação vira órfã na validação de FK e a Silver de transações
sai praticamente vazia — foi exatamente esse bug que o enunciado descreve (2.017 de 2.243 linhas
rejeitadas numa versão anterior do kit). tarifas não depende de nenhuma outra fonte, então processem
ela logo depois de clientes, sem deixar a fonte sem dependência atrasar a que tem uma dependência real
(transacoes). documentos é totalmente independente das outras três — não tem FK e ninguém depende
dela — e por isso fecha o pipeline, imediatamente antes de fundacao.finalizar_documentos.

Quando a leitura de um arquivo falhar por inteiro (encoding não aceito, schema drift, front-matter
ausente — qualquer exceção capturada no except), tratem o arquivo inteiro como quarentena: nunca
deixem a exc

## Passo 5 — carregar o modelo do Construtor

Qwen2.5-Coder-1.5B rodando dentro do notebook, em CPU. Baixa uma vez por sessão (~3 GB) e depois
responde em segundos por lacuna. Não depende de conta, de token nem de cota.

Enquanto baixa, leiam o esqueleto na célula seguinte: é o que vai ser preenchido.

In [58]:
print(agentes.carregar_modelo("Qwen/Qwen2.5-Coder-1.5B-Instruct"))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'modelo': 'Qwen/Qwen2.5-Coder-1.5B-Instruct', 'dispositivo': 'cuda:0', 'memoria_gb': 3.09, 'segundos_carga': 4.4}


In [59]:
print(agentes.ESQUELETO)


import fundacao

def pipeline(lake, contrato: dict) -> dict:
    contratos = contrato["fontes"]
    m = {"rows_in": 0, "rows_ok": 0, "rows_rejected": 0, "arquivos": 0, "corrigidos": 0, "drift": 0}
    ref = None
    for fonte in ___ORDEM_DAS_FONTES___:
        c = contratos[fonte]
        for r in fundacao.arquivos_pendentes(lake, c["padrao_arquivo"]):
            m["arquivos"] += 1
            try:
                df, correcoes = fundacao.ler(r["nome"], r["conteudo"], c)
                refs = ___REFERENCIAS_PARA_FK___
                ok, q, n_corr = fundacao.aplicar_contrato(df, c, refs)
                fundacao.gravar_silver(lake, fonte, ok, c["chave"], r["nome"])
                fundacao.gravar_quarentena(lake, fonte, r["nome"], q)
                fundacao.marcar_processado(lake, r["path"], fonte, "ok", len(ok), len(q))
                if fonte == "clientes" and ref is not None:
                    ref |= set(ok["cliente_id"])
                m["rows_in"] += len(df); m["rows_ok"] 

Um modelo de 1,5 bilhão de parâmetros não escreve um módulo inteiro que funcione. Escreve três decisões
específicas, se você perguntar uma de cada vez. O esqueleto fixo é o guardrail mais barato que existe:
troca "escreva o pipeline" por "complete esta lacuna", e o espaço de erro encolhe junto.

Isso não é limitação do exercício. É como se constrói agente de código em produção: contexto estreito,
formato fixo, verificação depois.

## Passo 6 — o Construtor escreve, e o kit testa antes de vocês confiarem

`construir` faz quatro coisas em sequência: gera as três lacunas, passa os guardrails estáticos, roda o
código num lakehouse descartável e compara o resultado com o esperado. Se reprovar, ele gera de novo
**com o diagnóstico na entrada**, até duas vezes.

Por que o teste de fumaça existe: guardrail estático não pega erro de lógica. Um código que processa
`transacoes` antes de `clientes` compila, executa, não levanta exceção nenhuma, e rejeita 2.017 linhas
das 2.243 porque toda transação virou órfã. Sem o teste, isso só aparece no harness, no fim da prática.

Cada tentativa leva de 45 a 90 segundos em CPU. Leiam o esqueleto enquanto roda.

In [60]:
r = agentes.construir(contrato_yaml, system_message, contratos, KIT, tentativas=2)
print("\nresultado:", "passou" if r["ok"] else "não passou", "· tentativas:", r["tentativas"])
g = {"codigo": r["codigo"]}


--- tentativa 1 de 2 ---
  ___ORDEM_DAS_FONTES___                      4.1s  sorted(contratos, key=lambda f: {"clientes": 1, "tarifas": 2, "transac
  ___REFERENCIAS_PARA_FK___                   2.8s  {"clientes": fundacao.referencia_clientes(lake)}
  ___O_QUE_FAZER_COM_O_ARQUIVO_INVALIDO___    6.8s  fundacao.gravar_quarentena(lake, fonte, r["nome"], None, str(e))
  teste de fumaça: passou {'rows_in': 2243, 'rows_ok': 2229, 'rows_rejected': 14, 'arquivos': 28, 'corrigidos': 6, 'drift': 0, 'duracao_s': 2.4}

resultado: passou · tentativas: 1


/content/dutos-do-q/kit/fundacao.py:114: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mudou = novo[(novo["vigente"] != docs["vigente"].fillna(False)) |
/content/dutos-do-q/kit/fundacao.py:115: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (novo["autoritativo"] != docs["autoritativo"].fillna(False))]


In [61]:
print(r["codigo"])


import fundacao

def pipeline(lake, contrato: dict) -> dict:
    contratos = contrato["fontes"]
    m = {"rows_in": 0, "rows_ok": 0, "rows_rejected": 0, "arquivos": 0, "corrigidos": 0, "drift": 0}
    ref = None
    for fonte in sorted(contratos, key=lambda f: {"clientes": 1, "tarifas": 2, "transacoes": 3, "documentos": 4}.get(f, 9)):
        c = contratos[fonte]
        for r in fundacao.arquivos_pendentes(lake, c["padrao_arquivo"]):
            m["arquivos"] += 1
            try:
                df, correcoes = fundacao.ler(r["nome"], r["conteudo"], c)
                refs = {"clientes": fundacao.referencia_clientes(lake)}
                ok, q, n_corr = fundacao.aplicar_contrato(df, c, refs)
                fundacao.gravar_silver(lake, fonte, ok, c["chave"], r["nome"])
                fundacao.gravar_quarentena(lake, fonte, r["nome"], q)
                fundacao.marcar_processado(lake, r["path"], fonte, "ok", len(ok), len(q))
                if fonte == "clientes" and ref is not No

## Passo 7 — Builders revisam antes de executar

Leiam o código. Três perguntas antes de apertar o botão:

1. A ordem das fontes está certa? Se `transacoes` vier antes de `clientes`, todas as transações viram órfãs.
2. As referências da FK estão sendo passadas só para `transacoes`?
3. Um arquivo quebrado vai inteiro para a quarentena, ou o erro engole o arquivo em silêncio?

O teste de fumaça já respondeu essas perguntas com números. A revisão de vocês é sobre o que fazer a
seguir: se o Construtor errou, **o que faltava na system message?** É essa a pergunta da ficha.

Se o esquadrão travar e o tempo apertar, a última célula adota a referência: vocês perdem os pontos da
geração, não a missão inteira.

In [62]:
if not r["ok"]:
    print("O Construtor não chegou lá em duas tentativas. O que ele errou:")
    for h in r["historico"]:
        print(" ", h.get("problemas") or h["diagnostico"].get("sintomas") or h["diagnostico"].get("erro"))
    print("\nAjustem a system message do Passo 4 e rodem o Passo 6 de novo, ou usem o plano B abaixo.")
else:
    print("Silver:", agentes.executar(r["codigo"], lake, {"fontes": contratos}))

Silver: {'rows_in': 0, 'rows_ok': 0, 'rows_rejected': 0, 'arquivos': 0, 'corrigidos': 0, 'drift': 0, 'duracao_s': 0.3}


In [63]:
# PLANO B do esquadrão travado (descomente as duas linhas e sigam para o Passo 8):
# g["codigo"] = agentes.codigo_de_referencia()
# print("Silver (referência):", agentes.executar(g["codigo"], lake, {"fontes": contratos}))

## Passo 8 — Red Team: leia a quarentena

A quarentena é o produto mais importante do duto. Uma linha rejeitada sem motivo legível é um chamado
de suporte na semana que vem.

In [64]:
print(lake.sql("""SELECT fonte, COUNT(*) linhas FROM silver.quarentena GROUP BY 1 ORDER BY 1""").to_string(index=False))
print()
print(lake.sql("""SELECT chave, motivo FROM silver.quarentena ORDER BY chave""").to_string(index=False))

     fonte  linhas
  clientes       1
transacoes      15

     chave                                                                                                                                            motivo
 *arquivo* arquivo rejeitado: schema drift: colunas obrigatórias ausentes ['valor']; colunas recebidas ['transacao_id', 'cliente_id', 'tipo', 'vlr', 'data']
 *arquivo*                                            arquivo rejeitado: 'contrato' codec can't decode byte 0x63 in position 0: nenhum encoding aceito: None
   linha 0                                                                                                                   data: data inválida: 16/06/2026
   linha 1                                                                                                                   data: data inválida: 15/06/2026
   linha 2                                                                                                                   data: data inválida: 03/06/2026


## Passo 9 — Gold: chunks e embeddings, só do que mudou

A Gold é o que o agente lê. Cada documento vira chunks, cada chunk vira um vetor, e cada chunk carrega
`vigente` e `autoritativo`.

O número a observar é `embeds_executados`. Na primeira execução ele é o total. Na segunda precisa ser
**zero**, porque nada mudou. Um duto que re-embeda tudo a cada execução funciona igual e custa dez vezes
mais, e é exatamente o tipo de decisão que ninguém revisa depois que entra em produção.

In [65]:
print("1ª execução:", dutos.gold(lake))
print("2ª execução:", dutos.gold(lake))

1ª execução: {'embeds_executados': 0, 'embeds_evitados': 53, 'chunks': 53, 'chunks_apagados': 0}
2ª execução: {'embeds_executados': 0, 'embeds_evitados': 53, 'chunks': 53, 'chunks_apagados': 0}


In [66]:
# a vigência em ação: a v1 da tabela de tarifas continua existindo, mas fora do índice do agente
print(lake.sql("""SELECT doc_id, versao, tipo, vigente, autoritativo FROM silver.documentos
                  WHERE doc_id IN ('prod-tabela-tarifas','mkt-blog-cdb','faq-antigo-tarifas-2023')
                  ORDER BY doc_id, versao""").to_string(index=False))

                 doc_id  versao       tipo  vigente  autoritativo
faq-antigo-tarifas-2023       1 faq-antigo     True         False
           mkt-blog-cdb       1  marketing     True         False
    prod-tabela-tarifas       1     tabela    False          True
    prod-tabela-tarifas       2     tabela     True          True


In [67]:
# e o efeito disso na busca: a pergunta sobre tarifa cai na versão certa
print(dutos.buscar(lake, "Qual a tarifa de saque em caixa eletrônico?", k=3)[["doc_id","versao","score"]].to_string(index=False))

             doc_id  versao    score
         ti-faq-app       1 0.706342
prod-tabela-tarifas       2 0.638448
prod-tabela-tarifas       2 0.478861


## Passo 10 — o Caos do professor (aos 40 minutos de prática)

Seis arquivos novos caem no inbox sem aviso: um CSV com coluna renomeada, um arquivo em latin-1, um
reenvio idêntico do que já foi processado, datas em dd/mm/aaaa, um comunicado legítimo e um comunicado
falso com tarifa de R$ 0,01 e data no futuro.

O duto de vocês roda igual. O que muda é se ele sobrevive.

**Só rode quando o professor mandar.**

In [68]:
print("caos no inbox:", dutos.soltar_caos(KIT, INBOX))

caos no inbox: ['clientes_novos_latin1.csv', 'com-novo-canal-whatsapp.md', 'com-tarifa-promocional.md', 'transacoes_datas_br.csv', 'transacoes_junho_drift.csv', 'transacoes_reenvio_duplicado.parquet']


In [69]:
print("Bronze:", dutos.bronze(lake, INBOX))
print("Silver:", agentes.executar(g["codigo"], lake, {"fontes": contratos}))
print("Gold  :", dutos.gold(lake))

Bronze: {'novos_arquivos': 0, 'total': 34, 'arquivos_na_pasta': 34}
Silver: {'rows_in': 0, 'rows_ok': 0, 'rows_rejected': 0, 'arquivos': 0, 'corrigidos': 0, 'drift': 0, 'duracao_s': 0.3}
Gold  : {'embeds_executados': 0, 'embeds_evitados': 53, 'chunks': 53, 'chunks_apagados': 0}


In [70]:
print("o comunicado falso entrou no índice?",
      lake.escalar("SELECT COUNT(*) FROM gold.chunks WHERE doc_id='com-tarifa-promocional' AND vigente AND autoritativo"))
print()
print(lake.sql("""SELECT arquivo, chave, motivo FROM silver.quarentena
                  WHERE arquivo LIKE '%drift%' OR arquivo LIKE '%promocional%'""").to_string(index=False))

o comunicado falso entrou no índice? 0

                   arquivo     chave                                                                                                                                            motivo
transacoes_junho_drift.csv *arquivo* arquivo rejeitado: schema drift: colunas obrigatórias ausentes ['valor']; colunas recebidas ['transacao_id', 'cliente_id', 'tipo', 'vlr', 'data']


## Passo 11 — o harness

Roda o ciclo completo duas vezes, mede as tabelas e devolve a nota. Ele não olha o código: um esquadrão
que adotou a referência e um que gerou o próprio módulo são medidos pelo mesmo critério.

In [71]:
# o harness mede desde o zero: lakehouse limpo e uma cópia intacta do inbox
lake_teste = Lake(str(KIT / "lakehouse_harness"))
lake_teste.zerar().criar_todas()
inbox_teste = dutos.preparar_inbox(KIT)

resultado = avaliacao.avaliar_m1(
    lake_teste, inbox_teste,
    lambda l: agentes.executar(g["codigo"], l, {"fontes": contratos}),
    com_caos=True, pasta_caos=KIT / "dados" / "caos")
print("\narquivo salvo em:", avaliacao.salvar(resultado, str(KIT / "resultados")))

/content/dutos-do-q/kit/fundacao.py:114: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mudou = novo[(novo["vigente"] != docs["vigente"].fillna(False)) |
/content/dutos-do-q/kit/fundacao.py:115: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (novo["autoritativo"] != docs["autoritativo"].fillna(False))]
/content/dutos-do-q/kit/fundacao.py:114: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set


MISSÃO 1 · DUTO BATCH — 100.0 pontos — OURO
  [OK  ] idempotência (2ª execução)                25 / 25   contagens iguais, embeddings na 2ª execução = 0
  [OK  ] integridade (contagens)                   20 / 20   {'clientes': (197, 197), 'transacoes': (2003, 2003), 'tarifas': (4, 4)}, doc_ids vigentes = 21 (esperado 21)
  [OK  ] qualidade (quarentena com motivo)       25.0 / 25   12/12 em quarentena, 0 inválidos vazaram para a Silver
  [OK  ] retrieval (top-3 vigente)                 10 / 10   +prod-tabela-tarifas +prod-cdb-quantum +com-lancamento-qi-cripto +cred-score-quantum +prod-credito-pessoal +rh-politica-home-office +prod-pix +prod-seguros +prod-conta-digital +ate
  [OK  ] índice limpo (só fonte autoritativa)      10 / 10   ruído no índice: nenhum; chunks de versão antiga ainda vigentes: 0
  [OK  ] SQL sobre a Silver                      10.0 / 10   +saldo da Marina +clientes válidos +tarifa de saque vigente em 02/2026 +transações da Marina
  [OK  ] CAOS: veneno bloqueado     

## Passo 12 — responder e entregar

Três perguntas sobre o que vocês acabaram de fazer. Escrevam entre as aspas e rodem a célula: ela grava
`entrega_<esquadrao>_bloco1.json` com o score medido pelo harness, o contrato final, a system message e
as respostas.

As perguntas são corrigidas pelo raciocínio, não pelo acerto. Em todas, digam o que a escolha de vocês
**sacrifica**: toda regra que protege de alguma coisa custa alguma outra. Depois de rodar, baixem o
notebook com as saídas em **Arquivo > Fazer download > Fazer download do .ipynb** e entreguem os dois.

In [73]:
respostas = {

"1. Qual lacuna do contrato vocês preencheram que mais mudou o resultado do harness? "
"Que evidência no dado levou a essa escolha, e o que essa regra rejeita que talvez fosse legítimo?":

"""
Preenchemos duas regras: `deposito_positivo` (`tipo != 'deposito' or valor > 0`) e `saida_negativa`
(`tipo not in ['saque','pix','cdb_aplicacao','compra_cartao'] or valor < 0`).

**Evidência:** no `transacoes.csv` do lote inicial, T990003 (`deposito`, valor `-300.00`) e T990004
(`deposito`, valor `-10.00`) são as únicas exceções ao padrão do arquivo inteiro — rodamos o contrato
contra as 2.011 linhas e confirmamos que todo `deposito` legítimo é positivo (min 0, exceto essas duas
plantadas) e todo `saque`/`pix`/`cdb_aplicacao`/`compra_cartao` é sempre negativo (entre -R$4.986 e
-R$20). Sem essa regra, essas duas linhas entrariam na Silver como depósitos negativos válidos.

**O que essa decisão rejeita que talvez fosse legítimo:** um estorno de depósito (o banco reverte um
depósito por erro operacional ou fraude) é legitimamente uma saída de caixa lançada com `tipo=deposito`
e `valor` negativo em alguns modelos de dados — e a nossa regra rejeitaria essa linha como se fosse
sujeira, quando na verdade seria um evento de negócio real que precisaria de um `tipo` próprio
(`estorno_deposito`) para não cair na quarentena. Validamos contra os dados de 2026 que temos e não
existe nenhuma linha assim no corpus, mas é o tipo de falso positivo que só aparece quando o produto
lança uma funcionalidade nova.

**Se a lacuna tivesse ficado em branco:** as quatro linhas com sinal errado (T990003, T990004, e o
equivalente do lado das saídas, se existisse) entrariam na Silver normalmente. O saldo calculado a
partir dessas transações ficaria sistematicamente errado, e é exatamente o tipo de erro que não dá
exceção nem warning — só aparece quando alguém soma o extrato e a conta não fecha, ou pior, quando o Q
informa um saldo errado para o cliente.

""",

"2. O que a system message precisou dizer para o Construtor acertar (ou o que faltou nela, se ele errou)? "
"Qual decisão deste pipeline vocês NÃO conseguiriam delegar a um agente, por melhor que fosse a instrução?":
"""
*Ainda em aberto — precisa ser respondido depois de rodar com o modelo de verdade (Qwen2.5-Coder-1.5B)
no Colab.* No ambiente em que preparamos este rascunho não temos acesso à internet necessária para
baixar o modelo (Hugging Face bloqueado), então validamos a lógica do contrato com o motor de regras
puro (`kit/contrato.py`) e com o backend `mock` do kit, que usa respostas canônicas fixas e **não
depende do texto da system message** — ou seja, não serve para responder esta pergunta.

Quando rodarem de verdade no Colab com `agentes.carregar_modelo(...)`, prestem atenção em duas frases
específicas da nossa system message e testem removê-las para ver se o Construtor erra:

- *"clientes vem primeiro porque transacoes tem chave estrangeira para cliente_id"* — sem essa frase, é
  bem provável que o modelo de 1,5B devolva uma ordem plausível mas errada (ex.: ordem alfabética), e o
  teste de fumaça vai acusar a maior parte das transações órfãs, igual ao bug descrito no enunciado.
- *"marquem o arquivo como processado com zero linhas nas duas contagens... e só depois somem 1 em
  drift e 1 em rows_rejected"* — essa é a parte mais fácil do modelo errar, porque pede pra ele lembrar
  que `ok` e `q` não existem dentro do bloco `except`. Se ele tentar usar `len(ok)` ali, o guardrail
  estático de sintaxe/nome indefinido deve reprovar na primeira tentativa.

Depois de rodar, substituam este parágrafo pela experiência real: o que o teste de fumaça acusou (se
acusou), o que vocês mudaram na system message, e qual decisão do pipeline (provavelmente a ordem das
fontes, que exige entender a FK) vocês não delegariam a um agente mesmo com instrução perfeita.
""",

"3. Qual linha da quarentena foi tratada errado, na opinião do esquadrão? "
"O que mudaria no contrato para corrigir, e o que essa mudança quebraria em outro lugar?":
"""
**Linha escolhida: `TF005` em `silver.quarentena` (fonte tarifas), motivo
`sem_sobreposicao_vigencia: sobrepõe TF001`.**

TF001 (saque, R$7,00, vigente de 2025-01-15 a 2026-01-09) e TF005 (saque, R$6,50, vigente de
2025-06-01 a 2025-08-01) se sobrepõem por inteiro — TF005 está cravado dentro do período de TF001.
O contrato rejeita TF005 (a segunda linha na ordenação por `vigencia_inicio`).

**Por que discordamos (ou pelo menos, por que a rejeição sozinha não resolve o problema):** o dado não
nos diz *qual das duas está errada*. Pode ser TF005 uma promoção temporária de dois meses (R$6,50 em
vez de R$7,00) que o time de produto lançou sem atualizar TF001 para refletir a pausa — nesse caso
TF001 é que deveria ter sido dividida em duas vigências (antes e depois da promoção), e TF005 é o dado
correto que faltou o "fechamento" da tarifa concorrente. O contrato manda a rejeitada para a quarentena
com motivo legível, mas não decide qual das duas é a verdade — só decide que as duas juntas não podem
estar vigentes ao mesmo tempo.

**O que mudaria no contrato, e o que isso quebraria:** poderíamos mudar a regra para, em vez de
descartar a mais recente, fechar automaticamente a vigência da mais antiga na data de início da mais
nova (regra de "a última vigência declarada vence"). Isso resolveria o caso da promoção, mas quebraria
o caso oposto: se TF005 fosse um erro de digitação (alguém subiu uma tarifa errada por engano com
vigência curta), a correção automática aplicaria esse erro como se fosse verdade, e ninguém revisaria.
Preferimos o comportamento atual — rejeitar e deixar visível na quarentena — porque um "quanto custava
o saque em julho de 2025" errado silenciosamente é pior do que um dado ausente que alguém do Red Team
vai notar e resolver com o time de produto antes do próximo `apply`.
""",

}

avaliacao.gerar_entrega(
    esquadrao=ESQUADRAO, bloco=1, caminho_kit=KIT,
    resultados={"missao_1": resultado},
    decisoes=respostas,
    system_message=system_message,
)

Entrega gravada em /content/dutos-do-q/entrega_esquadrao_00_bloco1.json
  esquadrão: esquadrao_00 · bloco 1
  missao_1: 100.0 pontos (OURO), total com Caos 120.0
  perguntas respondidas: 3 de 3

Agora faça: Arquivo > Fazer download > Fazer download do .ipynb
e entregue os DOIS arquivos.


'/content/dutos-do-q/entrega_esquadrao_00_bloco1.json'

Fim do Bloco 1.